---
layout: page
title: SpaceX
---

In [ ]:
""" import library and package dependencies """
from sys import path
path.insert(0, '../src')
from datetime import datetime
import pandas as pd
from IPython.display import display, Markdown

from data_loader import get_exchange_historical_data, get_exchange_data
from chart_format import StandardChart, LogChart, PercentileChart
from data_functions import Percentiles

SpaceX's IPO on the NASDAQ was in June 2026 under the ticker symbol SPCX. 

The initial months and years of price discovery may show significant volatility as both institutional and retail investors establish fair value for a company with long-term ambitions for Mars colonisation.

In [ ]:
"""
Load historical data
"""
sx_symbol = 'SPCX'

df = pd.DataFrame()

# For each previous year after 2026 get the historical data if not in local cache
current_year = datetime.now().year
for year in range(2026, current_year):
    from_date = str(year) + '-01-01'
    to_date = str(year) + '-12-31'
    local_file = '../data/' + sx_symbol.lower() + '_HistoricalData_' + str(year) + '.csv'
    df_year = get_exchange_historical_data(local_file, sx_symbol, from_date, to_date)
    df = pd.concat([df, df_year])

In [ ]:
"""
Load recent data for current year (always download / refresh)
"""
from_date = str(current_year) + '-01-01'
to_date = str(current_year) + '-12-31'
local_file = '../data/' + sx_symbol.lower() + '_HistoricalData_' + str(current_year) + '.csv'
# For current year always download the data and overwrite the local cache
df_year = get_exchange_data(local_file, sx_symbol, from_date, to_date)
df = pd.concat([df, df_year])

In [ ]:
""" Data Transformation """
data_column = 'Close'

# Ensure 'Date' column is timezone naive
df['Date'] = pd.to_datetime(df['Date'], utc=True)
df['Date'] = df['Date'].dt.tz_localize(None)
df.set_index('Date', inplace=True)

# Set 'last_index' to the last date with a valid value (so I can display 'as of <date>' in chart title)
last_index = df[data_column].last_valid_index()

In [ ]:
""" Show simple chart """
max_value = df[data_column].max()
top_limit = max_value + (100 - max_value % 100) # round UP to next 100

chart_params = {
    'chart_title': sx_symbol + ' Daily Price as of ' + last_index.strftime('%d %b %Y'),
    'chart_source': 'Source: finance.yahoo.com',
    'y_label': 'Price $ USD',
    'y_ticks': 50,
    'x_label': 'Date',
    'x_ticks': 1,
    'top_limit': top_limit,
    'data_column': data_column
}

standard_chart = StandardChart(**chart_params)
plt, colors = standard_chart.base_chart(df)

plt.plot(df.index, df[data_column], color=colors[0]['color'])

plt.show()

Over a long time period the price may appear to show exponential growth. Plotting the **same data** with a logarithmic y-axis scale helps visualize exponential trends. The data is the same, the only change is the y-axis uses a logarithmic scale.

In [ ]:
""" y-axis log scale """
log_chart_params = chart_params.copy()
log_chart_params['y_label'] = chart_params['y_label'] + ' (log scale)'
log_chart_params['y_ticks'] = [1, 2, 5, 10, 20, 50, 100, 200, 500, 1000]
del log_chart_params['top_limit']
log_chart = LogChart(**log_chart_params)
plt, colors = log_chart.base_chart(df)
plt.plot(df.index, df[data_column], color=colors[0]['color'])
plt.show()

In [ ]:
""" Calculate and print percentiles table """
""" Year over Year (YoY) change based on 252 trading days per year """
periods_per_year = 252
df['YoY'] = df[data_column].pct_change(periods=periods_per_year)*100
multi_years = [1, 3, 5, 10, 15]
percentiles = Percentiles(df, 'YoY', periods_per_year, multi_years)
df = percentiles.calculate_percentiles(df)

In [ ]:
percentiles.display_percentile_intro()

In [ ]:
yoy_chart_params = chart_params.copy()
yoy_chart_params['chart_title'] = sx_symbol + ' YoY Annual Price Change as of ' + last_index.strftime('%d %b %Y')
yoy_chart_params['data_column'] = 'YoY'
yoy_chart_params['x_ticks'] = 1
yoy_chart_params['y_label'] = 'YoY % Change'
yoy_chart_params['y_ticks'] = 50
yoy_chart_params['bottom_limit'] = -100
yoy_chart_params['top_limit'] = 1100
yoy_chart_params['color_index'] = 0
yoy_chart_params['legend_location'] = 'upper left'
percentile_chart = PercentileChart(percentiles.percentiles, percentiles.multi_years[3], **yoy_chart_params)
plt = percentile_chart.plot_percentiles(df)
plt.axhline(y=0, color='darkred', linestyle='-', linewidth=1)
plt.show()

In [ ]:
df_summary_percentiles = percentiles.display_summary(df, data_column)

In [ ]:
""" Change chart variables to plot zoomed in YoY chart """
yoy_chart_params['x_ticks'] = 1
yoy_chart_params['y_ticks'] = 10
yoy_chart_params['bottom_limit'] = -30
yoy_chart_params['top_limit'] = 140
percentile_chart = PercentileChart(percentiles.percentiles, percentiles.multi_years[3], **yoy_chart_params)
plt = percentile_chart.plot_percentiles(df)
plt.axhline(y=0, color='darkred', linestyle='-', linewidth=1)
plt.show()

In [ ]:
display(Markdown(f"""
> ℹ As trading history grows, the data will provide increasing confidence for a baseline CAGR between 
    { "{:,.0f}".format(df_summary_percentiles.loc[percentiles.multi_years[-2], 'CAGR']) }% 
    and { "{:,.0f}".format(df_summary_percentiles.loc[percentiles.multi_years[-1], 'CAGR']) }% 
    with uncertainty or risk that SPCX may show bursts of extreme growth with periods of volatility
    as the company executes on its long-term mission. Note this does not take inflation into consideration.
"""))

In [ ]:
%%capture
# Magic store dataframe to share with combo notebook
df_spcx = df
%store df_spcx